# Session 04: Policy Optimization and Actor-Critic Methods

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL5 Extended)  
**Level**: 🟡 Intermediate → 🔴 Advanced

---

## 📋 What You'll Learn

| Section | Topic | Level |
|---------|-------|---------|
| 1 | The Limits of Value-Based Methods | 🟡 |
| 2 | The Policy Gradient Theorem | 🟡 |
| 3 | REINFORCE: Monte Carlo Policy Gradient | 🔴 |
| 4 | The High Variance Problem & Baselines | 🔴 |
| 5 | Actor-Critic Architecture (A2C) | 🔴 |
| 6 | Modern Algorithms: TRPO & PPO | 🔴 |

In [ ]:
# Setup — Run this cell first!
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Hide TensorFlow warnings

import numpy as np
import tensorflow as as_tf
import tensorflow.keras as keras
from tensorflow.keras import layers, optimizers
import gymnasium as gym
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ All imports successful!')
print('   NumPy version:', np.__version__)
print('   TensorFlow version:', as_tf.__version__)
print('   Gymnasium version:', gym.__version__)

---

## §1 — The Limits of Value-Based Methods 🟡

In Session 03, we learned **Q-Learning** and **SARSA**.
In those algorithms, the agent literally learns a table (or network) of `Q(state, action)` values, and then chooses the action with the maximum value.

### Problem 1: Continuous Action Spaces
If your action is "Steering Wheel Angle", you have infinite possible actions (e.g., 14.5 degrees, 14.6 degrees). You cannot do `max_a Q(s, a)` over infinite actions!

### Problem 2: Stochastic Policies
Value-based methods are almost always *deterministic* (always pick the best action). 
But what if the best strategy is to be random? (Like Rock-Paper-Scissors). Q-Learning fails here.

### The Solution: Policy Optimization
Instead of predicting **Values**, we build a Neural Network that directly predicts **Probabilities for Actions**.

```
State (Vector)  --->  [ Neural Network ]  --->  Action Probabilities (e.g. Left: 0.2, Right: 0.8)
```

In [ ]:
# §1: Building a Policy Network (The Actor)

def build_policy_network(state_dim, action_dim):
    """A simple Policy Network for discrete actions."""
    model = keras.Sequential([
        layers.Dense(32, activation='relu', input_dim=state_dim),
        layers.Dense(32, activation='relu'),
        # Softmax ensures outputs are valid probabilities (sum to 1.0)
        layers.Dense(action_dim, activation='softmax') 
    ])
    return model

state_dim = 4  # e.g., CartPole (Cart Pos, Cart Vel, Pole Angle, Pole Vel)
action_dim = 2 # Left or Right

policy_net = build_policy_network(state_dim, action_dim)

# Let's see an example prediction
dummy_state = as_tf.convert_to_tensor([[0.0, 0.0, 0.0, 0.0]], dtype=as_tf.float32)
action_probs = policy_net(dummy_state).numpy()[0]

print(f"For a cart at rest in the center:")
print(f"  P(Move Left)  = {action_probs[0]:.2%}")
print(f"  P(Move Right) = {action_probs[1]:.2%}")
print(f"Notice they sum to perfectly 100%! The network is guessing randomly right now.")

---

## §2 — The Policy Gradient Theorem 🟡

How do we actually train this network? We can't use standard Supervised Learning because there are no "correct answers" (labels) given to us by the environment. 

We only have **Rewards**.

**The Logic:**
> “If an action resulted in a high reward, push its probability **UP**. If it resulted in a low reward, push its probability **DOWN**.”

**The Math (Policy Gradient Theorem):**
$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} [ \nabla_\theta \log \pi_\theta(a|s) \times G_t ]$

- $\nabla_\theta \log \pi_\theta$: How to change weights to make this action more likely.
- $G_t$: How good the total reward was from this time step onwards.

Since TensorFlow minimizes loss, we define our Loss as `-log(probability) * Return`.

In [ ]:
# §2: Calculating Discounted Returns

def calculate_returns(rewards, gamma=0.99):
    """
    If you receive rewards over 3 steps: [10, 10, 10]
    The 'Return' for step 0 is 10 + (0.99*10) + (0.99^2 * 10)
    """
    returns = np.zeros_like(rewards, dtype=np.float32)
    G = 0
    for t in reversed(range(len(rewards))):
        G = rewards[t] + gamma * G
        returns[t] = G
        
    return returns

sample_rewards = [1.0, 1.0, 1.0, 1.0, 1.0]
returns = calculate_returns(sample_rewards)

print("Rewards:", sample_rewards)
print("Returns:", [round(r, 2) for r in returns])
print("\nNotice that the earlier steps get ALL the credit for future rewards!")

---

## §3 — REINFORCE (Monte Carlo Policy Gradient) 🔴

The simplest complete PG algorithm is **REINFORCE**.

**Protocol:**
1. Start an episode. Use the Policy Network to pick actions until the episode ends.
2. Save all states, actions, and rewards in lists.
3. At the end (Monte Carlo), calculate the $G_t$ Return for every step.
4. Feed everything into TensorFlow. Compute the gradients $\log(P(a)) \times G$, and update the weights via Gradient Ascent.

### 👨‍💻 Run the code file `01_reinforce_cartpole.py` to see this in action on the CartPole environment!

---

## §4 — The High Variance Problem & Baselines 🔴

**The Flaw in REINFORCE:**
Because it waits until the end of an entire episode, the returns $G_t$ vary wildly. One episode might get a return of +500, and another might crash immediately and get +10. 

If **every** action in the good episode gets multiplied by +500, we accidentally reward the *bad* actions that just happened to exist in a good episode! This is called **High Variance**, and it makes training unstable.

### Baseline
Instead of multiplying by $G_t$, what if we multiplied by $(G_t - V(s))$ ?

$V(s)$ is what we *expected* to get at this state. If $G_t > V(s)$, the action was better than average! If $G_t < V(s)$, it was worse than average. This significantly reduces variance.

What figures out $V(s)$? **The Critic.**

---

## §5 — Advantage Actor-Critic (A2C) 🔴

Welcome to one of the most powerful paradigms in Deep RL: **Actor-Critic**.

We use **two** neural networks:
1. **The Actor (Policy)**: Learns *what to do* $\pi(a|s)$.
2. **The Critic (Value)**: Learns *how good a state is* $V(s)$.

### The Advantage
We don't wait for the end of the episode anymore. We calculate the Temporal Difference (TD) error instantly after taking a single step:

`Advantage = (Reward + γ * Critic(NextState)) - Critic(CurrentState)`

- If Advantage > 0: Tell the Actor to do that action MORE.
- If Advantage < 0: Tell the Actor to do that action LESS.

### 👨‍💻 Run the code file `02_actor_critic_cartpole.py` to see how much faster this learns than REINFORCE!

In [ ]:
# §5: Building the Critic Network

def build_critic_network(state_dim):
    """The Critic outputs a single continuous value estimating total future reward."""
    model = keras.Sequential([
        layers.Dense(32, activation='relu', input_dim=state_dim),
        layers.Dense(32, activation='relu'),
        # Linear activation for predicting continuous mathematical value V(s)
        layers.Dense(1, activation='linear') 
    ])
    return model

critic_net = build_critic_network(state_dim)
value = critic_net(dummy_state).numpy()[0][0]

print(f"The Critic estimates the value of this state is: {value:.3f}")
print(f"(It is initialized randomly, but it will learn to accurately predict returns!)")

---

## §6 — Modern Algorithms: TRPO & PPO 🔴

Basic Actor-Critic suffers from **Learning Rate sensitivity**. If the Actor takes a step that is too large, the policy gets "destroyed" and performance falls off a cliff.

### TRPO (Trust Region Policy Optimization)
Ensures that the new policy does not deviate from the old policy by more than a certain mathematical "distance" (KL Divergence). It's mathematically proven to monotonically improve, but it's computationally extremely heavy.

### PPO (Proximal Policy Optimization)
Introduced by OpenAI (used to train ChatGPT / RLHF!). 
PPO achieves the exact same stability as TRPO but using incredibly simple math:

`Ratio = Probability of Action under New Policy / Probability under Old Policy`

PPO literally just `clips` this ratio to be between `(1 - ε)` and `(1 + ε)` (usually 0.8 and 1.2).
This simple clip magically prevents the network from taking destructive updates.

### Most Modern Deep RL tasks (like autonomous driving) use PPO.

---

## 📝 Summary & Key Takeaways

1. **Policy Gradients** learn probabilities directly, allowing continuous actions and stochastic policies.
2. **REINFORCE** updates based on final episode returns, but suffers from high variance.
3. **Baselines** reduce this variance by comparing the return to an "expected" average.
4. **Actor-Critic** uses two networks: an Actor to choose actions, and a Critic to provide the baseline.
5. **PPO** is the modern industry standard for policy optimization, valued for its stability and simplicity.

### Next Steps:
1. Dive into `exercises/exercises.md` to mathematically prove these concepts.
2. Head to `portfolio/portfolio_component.md` to build and deploy an interface for observing Policy Optimization in action.